<div style="background:linear-gradient(135deg,#117A65 0%,#1E8449 100%);padding:40px 36px 32px 36px;border-radius:10px;margin-bottom:8px;">
  <p style="color:#A9DFBF;font-size:13px;margin:0 0 6px 0;letter-spacing:2px;">CURSO 8 · MÓDULO 3 · CLASE 9</p>
  <h1 style="color:white;font-size:36px;margin:0 0 10px 0;font-weight:700;">Ejercicios: QDA, Scores y Segmentación</h1>
  <p style="color:#A9DFBF;font-size:16px;margin:0 0 24px 0;font-style:italic;">Scores discriminantes · Error por clase · Caso marketing completo</p>
  <hr style="border-color:#52BE80;margin:0 0 20px 0;">
  <p style="color:#EAFAF1;font-size:13px;margin:0;">📌 <strong>Docente:</strong> Josef Rodriguez &nbsp;·&nbsp; 🟢 Básico · 🟡 Intermedio · 🔴 Avanzado</p>
</div>

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis, QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
np.set_printoptions(precision=4,suppress=True)
plt.rcParams.update({'figure.dpi':110,'font.size':11,'axes.spines.top':False,'axes.spines.right':False})
SEED=42; np.random.seed(SEED)
print('Setup listo')

---
## 🟢 Ejercicio 1 — Scores discriminantes e interpretación

1. Ajustar LDA a datos de 2 clases
2. Calcular scores = proyección en LD1
3. Graficar histogramas de scores por clase
4. Identificar las observaciones con menor confianza (cerca de la frontera)
5. ¿Qué relación tienen los scores con las probabilidades predichas?

In [ ]:
np.random.seed(SEED)
X_e1=np.vstack([np.random.multivariate_normal([-1.5,1.5],[[1.5,0.4],[0.4,1]],100),
                np.random.multivariate_normal([1.5,-1.5],[[1.5,0.4],[0.4,1]],100)])
y_e1=np.array([0]*100+[1]*100)
Xtr1,Xte1,ytr1,yte1=train_test_split(X_e1,y_e1,test_size=0.25,random_state=SEED)
# --- Tu código aquí ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
X_e1=np.vstack([np.random.multivariate_normal([-1.5,1.5],[[1.5,0.4],[0.4,1]],100),
                np.random.multivariate_normal([1.5,-1.5],[[1.5,0.4],[0.4,1]],100)])
y_e1=np.array([0]*100+[1]*100)
Xtr1,Xte1,ytr1,yte1=train_test_split(X_e1,y_e1,test_size=0.25,random_state=SEED)
lda1=LinearDiscriminantAnalysis(); lda1.fit(Xtr1,ytr1)
scores1=lda1.transform(Xte1)[:,0]
proba1=lda1.predict_proba(Xte1)[:,1]

fig,axes=plt.subplots(1,2,figsize=(13,4.5))
for cls,col in [(0,'#E74C3C'),(1,'#1E8449')]:
    axes[0].hist(scores1[yte1==cls],bins=20,alpha=0.7,color=col,label=f'Clase {cls}',density=True)
axes[0].axvline(0,color='gray',lw=2,linestyle='--',label='Frontera (score=0)')
axes[0].set(xlabel='Score LD1',title='Scores por clase'); axes[0].legend(); axes[0].grid(True,alpha=0.2)
axes[1].scatter(scores1,proba1,c=[('#E74C3C' if y==0 else '#1E8449') for y in yte1],s=30,alpha=0.7)
axes[1].axvline(0,color='gray',lw=1,linestyle='--'); axes[1].axhline(0.5,color='gray',lw=1,linestyle='--')
axes[1].set(xlabel='Score LD1',ylabel='P(clase 1)',title='Score vs Probabilidad')
axes[1].grid(True,alpha=0.2); plt.tight_layout(); plt.show()

# Obs con menor confianza (cerca de frontera)
incertidumbre=np.abs(scores1)
idx_inciertos=np.argsort(incertidumbre)[:5]
print(f'Accuracy: {accuracy_score(yte1,lda1.predict(Xte1)):.4f}')
print(f'\n5 observaciones menos confiables (score más cercano a 0):')
for i in idx_inciertos:
    print(f'  score={scores1[i]:+.3f}  P(1)={proba1[i]:.3f}  clase_real={yte1[i]}')
print('\nRelación: |score| mayor → probabilidad más extrema (más confianza)')

---
## 🟡 Ejercicio 2 — LDA vs QDA: análisis de covarianzas

1. Verificar el supuesto de Σ común entre clases
2. Comparar LDA vs QDA cuando el supuesto se cumple vs cuando no
3. Graficar las fronteras de decisión
4. Concluir cuándo usar cada uno

In [ ]:
np.random.seed(SEED)
# Caso A: covarianzas iguales (LDA OK)
Sigma_igual=np.array([[2,0.5],[0.5,1]])
X0_A=np.random.multivariate_normal([0,0],Sigma_igual,150)
X1_A=np.random.multivariate_normal([3,2],Sigma_igual,150)
# Caso B: covarianzas muy distintas (QDA mejor)
X0_B=np.random.multivariate_normal([0,0],[[3,0.5],[0.5,0.8]],150)
X1_B=np.random.multivariate_normal([3,2],[[0.8,0],[0,3]],150)
# --- Comparar LDA vs QDA en ambos casos ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
Sigma_igual=np.array([[2,0.5],[0.5,1]])
X0_A=np.random.multivariate_normal([0,0],Sigma_igual,150); X1_A=np.random.multivariate_normal([3,2],Sigma_igual,150)
X0_B=np.random.multivariate_normal([0,0],[[3,0.5],[0.5,0.8]],150); X1_B=np.random.multivariate_normal([3,2],[[0.8,0],[0,3]],150)
casos=[('Sigma iguales',X0_A,X1_A),('Sigma distintas',X0_B,X1_B)]
print(f'{"Caso":18s} {"Modelo":12s} {"AUC":>8s} {"Acc":>8s}')
print('─'*50)
for nombre,X0c,X1c in casos:
    Xc=np.vstack([X0c,X1c]); yc=np.array([0]*150+[1]*150)
    Xtr_c,Xte_c,ytr_c,yte_c=train_test_split(Xc,yc,test_size=0.25,random_state=SEED)
    for nm,m in [('LDA',LinearDiscriminantAnalysis()),('QDA',QuadraticDiscriminantAnalysis())]:
        m.fit(Xtr_c,ytr_c)
        auc=roc_auc_score(yte_c,m.predict_proba(Xte_c)[:,1])
        acc=accuracy_score(yte_c,m.predict(Xte_c))
        print(f'  {nombre:16s} {nm:12s} {auc:>8.4f} {acc:>8.4f}')
print('\nWhen Sigma igual: LDA ~= QDA. When Sigma muy distintas: QDA > LDA.')

---
## 🟡 Ejercicio 3 — Matriz de confusión y error por clase

Con 3 clases que se solapan parcialmente:
1. Ajustar LDA y QDA
2. Construir la matriz de confusión de cada uno
3. Identificar qué par de clases se confunde más
4. ¿Qué clase tiene mayor error? ¿Por qué?

In [ ]:
np.random.seed(SEED)
# Clases B y C muy cercanas (se solapan)
Sigma3=np.eye(3)
X_e3=np.vstack([np.random.multivariate_normal([-2,-2,0],Sigma3,120),
                np.random.multivariate_normal([0,1,-1],Sigma3*1.5,120),   # B y C solapan
                np.random.multivariate_normal([1,2,-0.5],Sigma3*1.5,120)])
y_e3=np.repeat(['A','B','C'],120)
# --- Construir matrices de confusión para LDA y QDA ---

In [ ]:
# ✅ SOLUCIÓN
np.random.seed(SEED)
Sigma3=np.eye(3)
X_e3=np.vstack([np.random.multivariate_normal([-2,-2,0],Sigma3,120),
                np.random.multivariate_normal([0,1,-1],Sigma3*1.5,120),
                np.random.multivariate_normal([1,2,-0.5],Sigma3*1.5,120)])
y_e3=np.repeat(['A','B','C'],120)
Xtr3,Xte3,ytr3,yte3=train_test_split(X_e3,y_e3,test_size=0.25,random_state=SEED)
for nm,m in [('LDA',LinearDiscriminantAnalysis()),('QDA',QuadraticDiscriminantAnalysis())]:
    m.fit(Xtr3,ytr3); y_pred3=m.predict(Xte3)
    cm3=confusion_matrix(yte3,y_pred3,labels=['A','B','C'])
    acc3=accuracy_score(yte3,y_pred3)
    print(f'\n{nm} — Accuracy: {acc3:.4f}')
    print(pd.DataFrame(cm3,index=['Real A','Real B','Real C'],columns=['Pred A','Pred B','Pred C']))
    print('\nError por clase:')
    for cls in ['A','B','C']:
        mask=np.array(yte3)==cls
        err=1-accuracy_score(np.array(yte3)[mask],np.array(y_pred3)[mask])
        print(f'  Clase {cls}: {err:.2%}')
print('\nB y C tienen mas errores porque sus medias estan mas cercanas.')

---
## 🔴 Ejercicio 4 — Pipeline de segmentación de clientes

Implementar `segment_customers(X_raw, y, feats, nuevos=None)` que:
1. Estandarice y ajuste LDA, QDA y Logística
2. Seleccione el mejor por accuracy
3. Reporte coeficientes LD del mejor modelo
4. Clasifique nuevos clientes con probabilidades
5. Grafique la proyección en espacio discriminante

In [ ]:
def segment_customers(X_raw, y, feats, nuevos=None):
    pass

np.random.seed(SEED); n_s=900
Sigma_s=np.array([[2,0.8,-0.3,0.4],[0.8,3,0.5,0],[-0.3,0.5,2,0.3],[0.4,0,0.3,1.5]])
X_s=np.vstack([np.random.multivariate_normal(mu,Sigma_s,n_s//3)
               for mu in [[-2,1,0.5,-1],[0,-1.5,-0.5,0.5],[2,1,-1,-0.5]]])
y_s=np.repeat(['Familiar','Joven_Urbano','Senior'],n_s//3)
feats_s=['freq_visitas','ticket_prom','compras_online','ratio_dcto']
nuevos_s=np.array([[-1.5,0.8,0.2,-0.8],[0.1,-1.2,1.5,0.6],[1.8,0.9,-1.1,-0.4]])
segment_customers(X_s, y_s, feats_s, nuevos_s)

In [ ]:
# ✅ SOLUCIÓN
def segment_customers(X_raw,y,feats,nuevos=None):
    sc=StandardScaler(); Xsc=sc.fit_transform(X_raw)
    Xtr,Xte,ytr,yte=train_test_split(Xsc,y,test_size=0.2,random_state=SEED)
    resultados={}
    print('Comparación de modelos:')
    for nm,m in [('LDA',LinearDiscriminantAnalysis()),
                  ('QDA',QuadraticDiscriminantAnalysis()),
                  ('Logistica',LogisticRegression(C=1e6,solver='lbfgs',max_iter=1000))]:
        m.fit(Xtr,ytr); acc=accuracy_score(yte,m.predict(Xte))
        resultados[nm]=(m,acc); print(f'  {nm}: Acc={acc:.4f}')
    mejor_nm=max(resultados,key=lambda k:resultados[k][1])
    mejor_m,mejor_acc=resultados[mejor_nm]
    print(f'\nMejor modelo: {mejor_nm} (Acc={mejor_acc:.4f})')
    print(classification_report(yte,mejor_m.predict(Xte)))
    if hasattr(mejor_m,'scalings_'):
        print('Coeficientes LD1 (importancia de features):')
        for nm,c in zip(feats,mejor_m.scalings_[:,0]):
            print(f'  {nm:20s}: {c:+.4f}')
    if hasattr(mejor_m,'transform'):
        X_proj=mejor_m.transform(Xte)
        fig,ax=plt.subplots(figsize=(7,5))
        clases=mejor_m.classes_; colores=['#1E8449','#2E86C1','#8E44AD']
        for cls,col in zip(clases,colores):
            m_=np.array(yte)==cls
            ax.scatter(X_proj[m_,0],X_proj[m_,1],s=22,alpha=0.7,color=col,label=cls)
        ax.set(xlabel='LD1',ylabel='LD2',title=f'Proyeccion {mejor_nm}')
        ax.legend(); ax.grid(True,alpha=0.2); plt.tight_layout(); plt.show()
    if nuevos is not None:
        nuevos_sc=sc.transform(nuevos); preds_n=mejor_m.predict(nuevos_sc)
        probs_n=mejor_m.predict_proba(nuevos_sc)
        print('\nNuevos clientes:')
        for i,(p,pr) in enumerate(zip(preds_n,probs_n)):
            print(f'  Cliente {chr(65+i)}: {p}  {dict(zip(mejor_m.classes_,pr.round(3)))}')

np.random.seed(SEED); n_s=900
Sigma_s=np.array([[2,0.8,-0.3,0.4],[0.8,3,0.5,0],[-0.3,0.5,2,0.3],[0.4,0,0.3,1.5]])
X_s=np.vstack([np.random.multivariate_normal(mu,Sigma_s,n_s//3)
               for mu in [[-2,1,0.5,-1],[0,-1.5,-0.5,0.5],[2,1,-1,-0.5]]])
y_s=np.repeat(['Familiar','Joven_Urbano','Senior'],n_s//3)
feats_s=['freq_visitas','ticket_prom','compras_online','ratio_dcto']
nuevos_s=np.array([[-1.5,0.8,0.2,-0.8],[0.1,-1.2,1.5,0.6],[1.8,0.9,-1.1,-0.4]])
segment_customers(X_s,y_s,feats_s,nuevos_s)

---
<div style="background:#117A65;color:white;padding:20px 24px;border-radius:8px;">
<strong>Módulo 3 completado ✓</strong><br>
LDA · QDA · Scores discriminantes · Segmentación de clientes<br>
Próximo: <strong>Módulo 4 — PCA</strong> · SVD · Varianza explicada · Scree plot<br>
<em>Docente: Josef Rodriguez · Curso 8</em>
</div>